# 📊 Aula 15 — Distribuição amostral no monitoramento de uma API

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados que monitora uma API** (o serviço que responde às requisições de um aplicativo). O time nunca olha todas as requisições: olha **amostras**. A Aula 15 responde à pergunta central desse trabalho: **se eu repetisse a amostragem muitas vezes, como se comportariam a média amostral $\bar{x}$ e a proporção amostral $\hat{p}$?** A resposta é a **distribuição amostral**.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`itertools.product`**: lista **todas** as amostras possíveis de uma população pequena;
- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: as "tabelas" da normal (`norm`) e da binomial (`binom`) dentro do computador.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
from itertools import product          # todas as combinações possíveis
import numpy as np                     # calculadora de números e sorteios
import matplotlib.pyplot as plt        # gráficos
from scipy.stats import norm, binom    # as "tabelas"

rng = np.random.default_rng(42)        # sorteador, semente 42
print("Tudo pronto!")

## 1. Todas as amostras possíveis (o exemplo da aula)

### A situação
É o exemplo da aula, que serve de "laboratório": a população é $\{1, 2, 3, 4, 5\}$, com $\mu = 3$ e $\sigma^2 = 2$. Tiramos **todas** as amostras de tamanho $n = 2$, **com reposição** — são $5 \times 5 = 25$ amostras — e calculamos a média de cada uma.

### A pergunta
Qual é a distribuição de $\bar{X}$? Quanto valem $E(\bar X)$ e $\text{Var}(\bar X)$?

### A ferramenta da aula
$$E(\bar X) = \mu \qquad \text{e} \qquad \text{Var}(\bar X) = \frac{\sigma^2}{n} .$$

### A resposta pela conta
$E(\bar X) = 3$ e $\text{Var}(\bar X) = \dfrac{2}{2} = 1$.

### O que o código faz
1. lista as 25 amostras e a média de cada uma;
2. monta a **tabela de distribuição** de $\bar X$: cada valor possível da média e a probabilidade dele (quantas amostras dão aquele valor, dividido por 25);
3. calcula a média e a variância de $\bar X$ a partir dessa tabela.

In [ ]:
populacao = [1, 2, 3, 4, 5]
amostras = list(product(populacao, repeat=2))                  # as 25 amostras (com reposição)
medias = np.array([np.mean(a) for a in amostras])

print("Algumas amostras:", amostras[:6], "...")
print(f"Total de amostras: {len(amostras)}\n")

print(" x̄     P(x̄)")
valores, contagens = np.unique(medias, return_counts=True)
for v, c in zip(valores, contagens):
    print(f" {v:.1f}   {c}/25 = {c / 25:.2f}")

print(f"\nE(x̄)   = {medias.mean():.2f}   (fórmula: μ = 3)")
print(f"Var(x̄) = {medias.var():.2f}   (fórmula: σ²/n = 2/2 = 1)")

plt.bar(valores, contagens / 25, width=0.4, color="steelblue", edgecolor="black")
plt.xlabel("Média da amostra (x̄)")
plt.ylabel("Probabilidade")
plt.title("Distribuição amostral de x̄: população {1,...,5}, n = 2")
plt.show()

**Como ler:** a média amostral pode valer de 1 a 5, mas os valores do meio são bem mais prováveis: $\bar x = 3$ sai em 5 das 25 amostras. A média de todas as médias é exatamente $\mu = 3$, e a variância caiu de $\sigma^2 = 2$ (dos valores individuais) para **1** (das médias). Médias variam **menos** que os valores individuais.

## 2. A média do tempo de resposta da API

### A situação
O tempo de resposta da API, em milissegundos (ms), tem média $\mu = 200$ ms e desvio padrão $\sigma = 200$ ms. A distribuição é bem **assimétrica**: a maioria das respostas é rápida, e algumas poucas demoram muito (é uma exponencial, da Aula 12). A cada minuto, o painel de monitoramento mostra a **média de 30 requisições**.

### A pergunta
1. Como se distribui essa média de 30 requisições? Ela também é assimétrica?
2. O painel dispara um alerta quando a média passa de **250 ms**. Se a API estiver funcionando normalmente, qual a chance de um **alarme falso**?

### A ferramenta da aula
- $E(\bar X) = \mu = 200$ e $\sigma_{\bar X} = \dfrac{\sigma}{\sqrt n} = \dfrac{200}{\sqrt{30}} \approx 36{,}5$ ms;
- pelo **Teorema do Limite Central (TLC)**, para $n$ grande, $\bar X$ é aproximadamente **normal**, mesmo que a população não seja.

### A resposta pela conta
$$P(\bar X > 250) \approx P\left(Z > \frac{250 - 200}{36{,}5}\right) = P(Z > 1{,}37) \approx 0{,}0855 .$$

### O que o código faz
1. sorteia **100.000 amostras** de cada tamanho ($n = 1$, $5$ e $30$) do tempo de resposta e calcula a média de cada uma;
2. desenha um histograma das médias para cada $n$, lado a lado;
3. para $n = 30$, compara a chance de alarme falso pela normal e pela simulação.

In [ ]:
mu, sigma = 200, 200                                           # ms

fig, eixos = plt.subplots(1, 3, figsize=(14, 4))
for eixo, n in zip(eixos, (1, 5, 30)):
    medias = rng.exponential(mu, size=(100_000, n)).mean(axis=1)   # 100.000 médias de n requisições
    eixo.hist(medias, bins=80, range=(0, 800), density=True, color="steelblue", edgecolor="black")
    eixo.set_title(f"n = {n}: média {medias.mean():.0f} ms, desvio {medias.std():.1f} ms")
    eixo.set_xlabel("Média das n requisições (ms)")
plt.show()

erro_padrao = sigma / np.sqrt(30)
print(f"σ/√n para n = 30: {erro_padrao:.1f} ms")
print(f"P(x̄ > 250)  pela normal (TLC): {1 - norm.cdf((250 - mu) / erro_padrao):.4f}")
print(f"P(x̄ > 250)  na simulação:      {(medias > 250).mean():.4f}")

**Como ler:**
- Com $n = 1$ (uma requisição só), o histograma tem o formato da população: muito assimétrico.
- Com $n = 5$, as médias já se concentram mais perto de 200 e a assimetria diminui.
- Com $n = 30$, o histograma fica com **formato de sino**, centrado em 200 ms e com desvio de cerca de **36,5 ms** ($= 200/\sqrt{30}$). É o TLC em ação.
- A chance de alarme falso é de cerca de **8,5%** pela normal; a simulação dá um pouco mais (cerca de 9%), porque com $n = 30$ ainda sobra um resto da assimetria da população.

**Conclusão para o time:** com o limite de 250 ms, o painel daria um alarme falso a cada 11 ou 12 minutos, em média. O limite está baixo demais, ou o painel deveria usar amostras maiores (que diminuem $\sigma/\sqrt n$).

## 3. A proporção de requisições lentas

### A situação
Num dia normal, **30%** das requisições de um serviço antigo voltam com aviso de lentidão. O time sorteia requisições do log e calcula $\hat p$ = proporção de avisos na amostra.

### A pergunta
Qual a chance de $\hat p$ ficar a **menos de 0,01** (um ponto percentual) da proporção verdadeira $p = 0{,}30$, com uma amostra de **10** requisições? E de **2.000**?

### A ferramenta da aula
$$\hat p \approx N\!\left(p,\ \frac{p(1-p)}{n}\right), \qquad \text{então} \qquad P(|\hat p - p| < 0{,}01) \approx P\!\left(|Z| < \frac{0{,}01}{\sqrt{p(1-p)/n}}\right).$$

É o mesmo cálculo do exemplo da aula com $p = 30\%$ e $n = 10$.

### A resposta pela conta
- $n = 10$: $\sqrt{0{,}21/10} \approx 0{,}145$, então $P(|Z| < 0{,}069) \approx 0{,}055$;
- $n = 2000$: $\sqrt{0{,}21/2000} \approx 0{,}0102$, então $P(|Z| < 0{,}976) \approx 0{,}671$.

### O que o código faz
Para cada tamanho de amostra, calcula a probabilidade de três jeitos: pela **normal** (fórmula da aula), pela **binomial exata** ($n\hat p$ é o número de avisos, que é binomial) e por **simulação** de 100.000 amostras.

In [ ]:
p, tolerancia = 0.30, 0.01

for n in (10, 2000):
    dp = np.sqrt(p * (1 - p) / n)                                  # desvio padrão de p̂
    pela_normal = 2 * norm.cdf(tolerancia / dp) - 1                # P(|Z| < 0,01/dp)
    k = np.arange(n + 1)                                           # valores possíveis do nº de avisos
    dentro = np.abs(k / n - p) < tolerancia                        # quais deixam p̂ perto de p
    exata = binom.pmf(k[dentro], n, p).sum()
    p_chapeu = rng.binomial(n, p, size=100_000) / n                # 100.000 proporções simuladas
    print(f"n = {n:4d}: desvio de p̂ = {dp:.4f}   normal = {pela_normal:.4f}   "
          f"binomial exata = {exata:.4f}   simulação = {(np.abs(p_chapeu - p) < tolerancia).mean():.4f}")

**Como ler:**
- Com **2.000** requisições, $\hat p$ fica a menos de um ponto percentual de $p$ em cerca de **67%** das amostras, e a normal (0,671) concorda bem com a conta exata (0,659).
- Com só **10** requisições, a normal dá 5,5%, mas a conta exata dá **26,7%**! O motivo: com $n = 10$, $\hat p$ só pode valer 0; 0,1; 0,2; ...; e a única forma de ficar a menos de 0,01 de 0,30 é acertar **exatamente** 0,3 (3 avisos em 10), o que tem chance $P(X = 3) \approx 0{,}267$. A normal não "enxerga" esses saltos.

**Lição:** a aproximação normal de $\hat p$ é para **amostras grandes** (a mesma regra da aproximação normal à binomial: $np$ e $n(1-p)$ grandes). Com $n = 10$, a pergunta deve ser respondida pela binomial.

## Para fechar

Neste notebook, o time de monitoramento usou a **distribuição amostral** da Aula 15:

- listando **todas** as amostras possíveis, viu que $E(\bar X) = \mu$ e $\text{Var}(\bar X) = \sigma^2/n$;
- viu a média de uma população **assimétrica** ganhar formato de **sino** quando $n$ cresce, e usou isso para calcular a chance de alarme falso;
- usou $\hat p \approx N(p,\ p(1-p)/n)$ para a proporção, e viu que a aproximação só é boa com amostras grandes.

Na **Aula 16**, o **Teorema do Limite Central** ganha destaque e começamos a discutir o que torna um **estimador** bom.